# H-04: Cross-Category Success Rate Uniformity

**Question:** Does the agent succeed equally across all fault categories — or does it systematically fail on specific ones?

| | |
|---|---|
| **H₀ (Null):** | Success rates are the same across all fault categories. |
| **Hₐ (Alternative):** | At least one category has a significantly different success rate. |
| **Certification rule:** | If rates are non-uniform, the weakest category is flagged. |
| **Metrics:** | fault_detection, fault_mitigation, rai_compliance, security_compliance |
| **Method:** | Chi-Square test on contingency table (pooled counts per category) |

### H-04 vs H-02 vs H-03
| Test | Question | Data type | Method |
|------|----------|-----------|--------|
| H-02 | What's the guaranteed floor for each category? | Binary rates | Wilson CI |
| H-03 | Do continuous metrics differ across categories? | Continuous | Kruskal-Wallis |
| **H-04** | Do success *rates* differ across categories? | Binary counts | Chi-Square |

### Aggregation
- **Pooled counts:** Sub-fault successes/trials summed per category for the contingency table
- **Within-category heterogeneity:** Chi-square among sub-faults flags internal disparity
- **Equal-weight rate:** Average of sub-fault rates, reported alongside for H-02 consistency


In [1]:
import sys, os
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def fuzzy_match(s1, s2, threshold=0.6):
    if s1 is None or s2 is None:
        return False
    s1, s2 = s1.lower().strip(), s2.lower().strip()
    if s1 == s2 or s1 in s2 or s2 in s1:
        return True
    return SequenceMatcher(None, s1, s2).ratio() >= threshold

def is_correct_detection(run):
    q = run["quantitative"]
    if q.get("fault_detected") != "Yes":
        return False
    return fuzzy_match(q.get("injected_fault_name"), q.get("detected_fault_type"))

def build_subfault_detection_counts(runs):
    """Count correct detections per sub-fault (fuzzy match)."""
    grouped = defaultdict(lambda: [0, 0])
    for run in runs:
        fname = run["fault_name"]
        grouped[fname][1] += 1
        if is_correct_detection(run):
            grouped[fname][0] += 1
    return {k: tuple(v) for k, v in grouped.items()}

def build_subfault_counts(runs, field, value="Yes", source="quantitative"):
    """Count successes per sub-fault for a binary field."""
    grouped = defaultdict(lambda: [0, 0])
    for run in runs:
        fname = run["fault_name"]
        grouped[fname][1] += 1
        if source == "quantitative":
            val = run["quantitative"].get(field)
        else:
            val = run["qualitative"].get(field)
        if val == value:
            grouped[fname][0] += 1
    return {k: tuple(v) for k, v in grouped.items()}

# Load all runs
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    det = sum(1 for r in runs if r['quantitative'].get('fault_detected') == 'Yes')
    mit = sum(1 for r in runs if r['quantitative'].get('fault_mitigated') == 'Yes')
    print(f"  {cat}: {total} runs, {det} detected ({det/total:.0%}), {mit} mitigated ({mit/total:.0%})")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%), 45 mitigated (75%)
  network_fault: 90 runs, 39 detected (43%), 14 mitigated (16%)
  resource_fault: 90 runs, 70 detected (78%), 50 mitigated (56%)


---
## Step 1: Per Sub-Fault Detection Rates

Preview how detection rates vary across sub-faults and categories.


In [2]:
print("Detection rates per sub-fault:")
print("=" * 75)

for cat, runs in categories.items():
    sf_counts = build_subfault_detection_counts(runs)
    total_s = sum(s for s, n in sf_counts.values())
    total_n = sum(n for s, n in sf_counts.values())
    print(f"\n{cat}: {total_s}/{total_n} = {total_s/total_n:.0%}")
    for fn, (s, n) in sorted(sf_counts.items()):
        print(f"  {fn:25s}: {s:2d}/{n:2d} = {s/n:.0%}")


Detection rates per sub-fault:

application_fault: 51/60 = 85%
  container-kill           : 28/30 = 93%
  pod-delete               : 23/30 = 77%

network_fault: 39/90 = 43%
  pod-dns-error            : 13/30 = 43%
  pod-network-corruption   : 14/30 = 47%
  pod-network-loss         : 12/30 = 40%

resource_fault: 70/90 = 78%
  disk-fill                : 25/30 = 83%
  pod-cpu-hog              : 25/30 = 83%
  pod-memory-hog           : 20/30 = 67%


---
## Step 2: Run H-04 — Fault Detection Rate Uniformity

Chi-square test on contingency table: [detected, not-detected] × [categories].
H₀: Detection rates are the same across all fault categories.


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h04_success_rate_uniformity import run_uniformity_test

# Build input: {category: {sub_fault: (successes, trials)}}
det_counts = {}
for cat, runs in categories.items():
    det_counts[cat] = build_subfault_detection_counts(runs)

h04_det = run_uniformity_test(det_counts, metric_name="fault_detection_rate")

print(f"H-04: {h04_det.hypothesis_name}")
print(f"Metric: {h04_det.metric_name}")
print(f"Overall: {h04_det.overall_assessment}")
print("=" * 85)

# Per-category details
print("\n--- Per-Category Details ---")
for c in h04_det.per_category:
    het_lbl = " \u26a0\ufe0f HETEROGENEOUS" if c.within_heterogeneous else " (homogeneous)"
    print(f"\n  {c.category} ({c.successes}/{c.trials} = {c.rate:.0%}{het_lbl}):")
    print(f"    Pooled rate: {c.rate:.1%}    Equal-weight rate: {c.equal_weight_rate:.1%}")
    if c.within_p is not None:
        print(f"    Within-cat test: p={c.within_p:.4f}")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: {sf.successes:2d}/{sf.trials:2d} = {sf.rate:.0%}  "
              f"Wilson [{sf.wilson_lower:.1%}, {sf.wilson_upper:.1%}]")

# Omnibus test
stat_lbl = f"χ²={h04_det.statistic:.4f}" if h04_det.statistic else "Fisher"
print(f"\n--- Omnibus: {h04_det.test_used} ---")
print(f"  {stat_lbl},  p = {h04_det.p_value:.6f}")
print(f"  Significant: {h04_det.significant}")
print(f"  Weakest category: {h04_det.weakest_category}")

if h04_det.warnings:
    print(f"\nWarnings:")
    for w in h04_det.warnings:
        print(f"  - {w}")


H-04: Cross-Category Success Rate Uniformity
Metric: fault_detection_rate
Overall: non_uniform_rates

--- Per-Category Details ---

  application_fault (51/60 = 85% (homogeneous)):
    Pooled rate: 85.0%    Equal-weight rate: 85.0%
    Within-cat test: p=0.1455
      container-kill           : 28/30 = 93%  Wilson [78.7%, 98.2%]
      pod-delete               : 23/30 = 77%  Wilson [59.1%, 88.2%]

  network_fault (39/90 = 43% (homogeneous)):
    Pooled rate: 43.3%    Equal-weight rate: 43.3%
    Within-cat test: p=0.8731
      pod-dns-error            : 13/30 = 43%  Wilson [27.4%, 60.8%]
      pod-network-corruption   : 14/30 = 47%  Wilson [30.2%, 63.9%]
      pod-network-loss         : 12/30 = 40%  Wilson [24.6%, 57.7%]

  resource_fault (70/90 = 78% (homogeneous)):
    Pooled rate: 77.8%    Equal-weight rate: 77.8%
    Within-cat test: p=0.2005
      disk-fill                : 25/30 = 83%  Wilson [66.4%, 92.7%]
      pod-cpu-hog              : 25/30 = 83%  Wilson [66.4%, 92.7%]
      p

---
## Step 3: Run H-04 — Fault Mitigation Rate Uniformity


In [4]:
mit_counts = {}
for cat, runs in categories.items():
    mit_counts[cat] = build_subfault_counts(runs, "fault_mitigated", "Yes")

h04_mit = run_uniformity_test(mit_counts, metric_name="fault_mitigation_rate")

print(f"H-04: {h04_mit.metric_name}  |  {h04_mit.overall_assessment}")
print("=" * 85)
for c in h04_mit.per_category:
    het_lbl = " \u26a0\ufe0f" if c.within_heterogeneous else ""
    print(f"  {c.category:20s}: {c.successes:2d}/{c.trials:2d} = {c.rate:.0%}  EW={c.equal_weight_rate:.0%}{het_lbl}")

stat_lbl = f"χ²={h04_mit.statistic:.4f}" if h04_mit.statistic else "Fisher"
print(f"\n  {h04_mit.test_used}: {stat_lbl}, p={h04_mit.p_value:.6f}, sig={h04_mit.significant}")
print(f"  Weakest: {h04_mit.weakest_category}")


H-04: fault_mitigation_rate  |  non_uniform_rates
  application_fault   : 45/60 = 75%  EW=75%
  network_fault       : 14/90 = 16%  EW=16%
  resource_fault      : 50/90 = 56%  EW=56%

  chi_square: χ²=57.2869, p=0.000000, sig=True
  Weakest: network_fault


---
## Step 4: Run H-04 — RAI Compliance Rate Uniformity


In [5]:
rai_counts = {}
for cat, runs in categories.items():
    rai_counts[cat] = build_subfault_counts(runs, "rai_check_status", "Passed", source="qualitative")

h04_rai = run_uniformity_test(rai_counts, metric_name="rai_compliance_rate")

print(f"H-04: {h04_rai.metric_name}  |  {h04_rai.overall_assessment}")
print("=" * 85)
for c in h04_rai.per_category:
    het_lbl = " \u26a0\ufe0f" if c.within_heterogeneous else ""
    print(f"  {c.category:20s}: {c.successes:2d}/{c.trials:2d} = {c.rate:.0%}  EW={c.equal_weight_rate:.0%}{het_lbl}")

stat_lbl = f"χ²={h04_rai.statistic:.4f}" if h04_rai.statistic else "Fisher"
print(f"\n  {h04_rai.test_used}: {stat_lbl}, p={h04_rai.p_value:.6f}, sig={h04_rai.significant}")
print(f"  Weakest: {h04_rai.weakest_category}")


H-04: rai_compliance_rate  |  uniform_rates
  application_fault   : 60/60 = 100%  EW=100%
  network_fault       : 90/90 = 100%  EW=100%
  resource_fault      : 90/90 = 100%  EW=100%

  chi_square: Fisher, p=1.000000, sig=False
  Weakest: application_fault


---
## Step 5: Run H-04 — Security Compliance Rate Uniformity


In [6]:
sec_counts = {}
for cat, runs in categories.items():
    sec_counts[cat] = build_subfault_counts(runs, "security_compliance_status", "Passed", source="qualitative")

h04_sec = run_uniformity_test(sec_counts, metric_name="security_compliance_rate")

print(f"H-04: {h04_sec.metric_name}  |  {h04_sec.overall_assessment}")
print("=" * 85)
for c in h04_sec.per_category:
    het_lbl = " \u26a0\ufe0f" if c.within_heterogeneous else ""
    print(f"  {c.category:20s}: {c.successes:2d}/{c.trials:2d} = {c.rate:.0%}  EW={c.equal_weight_rate:.0%}{het_lbl}")

stat_lbl = f"χ²={h04_sec.statistic:.4f}" if h04_sec.statistic else "Fisher"
print(f"\n  {h04_sec.test_used}: {stat_lbl}, p={h04_sec.p_value:.6f}, sig={h04_sec.significant}")
print(f"  Weakest: {h04_sec.weakest_category}")


H-04: security_compliance_rate  |  uniform_rates
  application_fault   :  0/60 = 0%  EW=0%
  network_fault       :  0/90 = 0%  EW=0%
  resource_fault      :  0/90 = 0%  EW=0%

  chi_square: Fisher, p=1.000000, sig=False
  Weakest: application_fault


---
## Step 6: Summary & Interpretation


In [7]:
print("H-04 Cross-Category Success Rate Uniformity \u2014 Summary")
print("=" * 85)

results = {
    "fault_detection_rate": h04_det,
    "fault_mitigation_rate": h04_mit,
    "rai_compliance_rate": h04_rai,
    "security_compliance_rate": h04_sec,
}

for metric, r in results.items():
    sig_lbl = "\u274c NON-UNIFORM" if r.significant else "\u2705 UNIFORM"
    stat_lbl = f"\u03c7\u00b2={r.statistic:.2f}" if r.statistic else "Fisher"
    print(f"\n  {metric}: {sig_lbl}  ({r.test_used}: {stat_lbl}, p={r.p_value:.4f})")
    print(f"    Rates: ", end="")
    print("  ".join(f"{cat.replace('_fault','')}={rate:.0%}" for cat, rate in r.per_category_rates.items()))
    if r.significant:
        print(f"    Weakest: {r.weakest_category}")
    het_cats = [c.category for c in r.per_category if c.within_heterogeneous]
    if het_cats:
        print(f"    \u26a0\ufe0f Within-category heterogeneity: {', '.join(het_cats)}")

print("\n" + "=" * 85)
print("\nInterpretation:")
print("  - NON-UNIFORM: The agent's success rate varies significantly across categories.")
print("    This means the agent is reliable for some fault types but unreliable for others.")
print("  - UNIFORM: No statistical evidence that success rates differ across categories.")


H-04 Cross-Category Success Rate Uniformity — Summary

  fault_detection_rate: ❌ NON-UNIFORM  (chi_square: χ²=36.12, p=0.0000)
    Rates: application=85%  network=43%  resource=78%
    Weakest: network_fault

  fault_mitigation_rate: ❌ NON-UNIFORM  (chi_square: χ²=57.29, p=0.0000)
    Rates: application=75%  network=16%  resource=56%
    Weakest: network_fault

  rai_compliance_rate: ✅ UNIFORM  (chi_square: Fisher, p=1.0000)
    Rates: application=100%  network=100%  resource=100%

  security_compliance_rate: ✅ UNIFORM  (chi_square: Fisher, p=1.0000)
    Rates: application=0%  network=0%  resource=0%


Interpretation:
  - NON-UNIFORM: The agent's success rate varies significantly across categories.
    This means the agent is reliable for some fault types but unreliable for others.
  - UNIFORM: No statistical evidence that success rates differ across categories.


---
## Step 7: JSON Output


In [8]:
output = {
    "hypothesis_id": "H-04",
    "hypothesis_name": "Cross-Category Success Rate Uniformity",
    "null_hypothesis": "Success rates are the same across all fault categories.",
    "alt_hypothesis": "At least one category has a significantly different success rate.",
    "certification_rule": "If rates are non-uniform, the weakest category is flagged.",
    "hypothesis_metadata": {
        "method": "Chi-Square test on contingency table (pooled counts per category)",
        "aggregation": "pooled sub-fault counts per category",
        "alpha": 0.05,
        "categories": list(categories.keys()),
        "n_runs_per_category": {cat: len(runs) for cat, runs in categories.items()},
        "mode": "both (always active)",
    },
    "results": {},
}

for metric, r in results.items():
    output["results"][metric] = {
        "overall_assessment": r.overall_assessment,
        "test_used": r.test_used,
        "statistic": r.statistic,
        "p_value": r.p_value,
        "significant": r.significant,
        "weakest_category": r.weakest_category,
        "per_category": [
            {
                "category": c.category,
                "successes": c.successes,
                "trials": c.trials,
                "rate": c.rate,
                "equal_weight_rate": c.equal_weight_rate,
                "within_heterogeneous": c.within_heterogeneous,
                "within_p": c.within_p,
            }
            for c in r.per_category
        ],
    }

print(json.dumps(output, indent=2))


{
  "hypothesis_id": "H-04",
  "hypothesis_name": "Cross-Category Success Rate Uniformity",
  "null_hypothesis": "Success rates are the same across all fault categories.",
  "alt_hypothesis": "At least one category has a significantly different success rate.",
  "certification_rule": "If rates are non-uniform, the weakest category is flagged.",
  "hypothesis_metadata": {
    "method": "Chi-Square test on contingency table (pooled counts per category)",
    "aggregation": "pooled sub-fault counts per category",
    "alpha": 0.05,
    "categories": [
      "application_fault",
      "network_fault",
      "resource_fault"
    ],
    "n_runs_per_category": {
      "application_fault": 60,
      "network_fault": 90,
      "resource_fault": 90
    },
    "mode": "both (always active)"
  },
  "results": {
    "fault_detection_rate": {
      "overall_assessment": "non_uniform_rates",
      "test_used": "chi_square",
      "statistic": 36.125,
      "p_value": 0.0,
      "significant": true,
 